# IN6227-Assignment-1 | Variant-1
**Binary classification: Logistic Regression vs Random Forest**

Student: Zhang Huanzun · Matriculation number: **G2609063G**

本 Notebook 包含完整作业流程。Colab 选择 **Runtime → Run all**，提示时同时上传 `train.csv` 和 `test.csv`。使用 CPU 即可。
本地运行时将两个 CSV 放在 Notebook 同目录或 `dataset/` 文件夹；Colab 运行时按提示上传。

**Evaluation contract:** the provided train/test split is preserved. All data-dependent preprocessing and hyperparameter selection are fitted within training folds. The test set is used only after configurations and the probability threshold (0.5) are fixed. `yes` is the positive class. Average precision (AP) is preselected to summarize precision across recall levels; ROC-AUC and threshold metrics provide complementary views.

The report must be at most two pages, include full name, matric number, the assignment and variant identifiers, and the GitHub repository link. Review the decisions and results before submission.

In [ ]:
from pathlib import Path
import json, time, platform, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay)
from sklearn.exceptions import ConvergenceWarning
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, 'to_string') else value)

SEED = 42
OUT = Path('results')
OUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 115, 'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False, 'savefig.bbox': 'tight'})
VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scikit-learn': sklearn.__version__}
print('Runtime versions:', VERSIONS)
print('CPU runtime; no GPU or PyTorch required.')

## 1. Load the supplied split / 上传与读取
Upload both original filenames when prompted. If already present, they are reused. To run locally, put the CSVs beside the notebook or in a `dataset` subdirectory. No data are downloaded from an external dataset source.

In [ ]:
candidate_dirs = [Path('.'), Path('dataset'), Path('/content')]
DATA_DIR = next((d for d in candidate_dirs
                 if (d/'train.csv').is_file() and (d/'test.csv').is_file()), None)
if DATA_DIR is None:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Place train.csv and test.csv beside this notebook or in dataset/.')
    print('Select train.csv AND test.csv from your dataset folder.')
    files.upload()
    DATA_DIR = Path('.')
    if not all((DATA_DIR/f).is_file() for f in ['train.csv', 'test.csv']):
        raise FileNotFoundError('Both train.csv and test.csv must be uploaded with their original filenames.')

train_raw = pd.read_csv(DATA_DIR/'train.csv')
test_raw = pd.read_csv(DATA_DIR/'test.csv')
TARGET = 'label'
assert TARGET in train_raw.columns and TARGET in test_raw.columns
assert train_raw.columns.tolist() == test_raw.columns.tolist(), 'Train/test schemas differ.'
SOURCE_HASHES = {f: hashlib.sha256((DATA_DIR/f).read_bytes()).hexdigest()
                 for f in ['train.csv', 'test.csv']}
print('Original shapes:', {'train': train_raw.shape, 'test': test_raw.shape})
display(train_raw.head())
# Test features/labels are not used to choose models or hyperparameters.

## 2. Training-data exploration / 只用训练数据进行探索
We inspect types, missingness, duplicates, imbalance, numerical ranges and correlations. A 1.5-IQR flag is a diagnostic, not proof of a data error. The column names alone do not establish a real-world domain or valid physical limits.

In [ ]:
numeric_cols = train_raw.drop(columns=TARGET).select_dtypes(include='number').columns.tolist()
categorical_cols = [c for c in train_raw.columns if c not in numeric_cols + [TARGET]]
schema = pd.DataFrame({'dtype': train_raw.dtypes.astype(str),
                       'missing': train_raw.isna().sum(),
                       'distinct_nonmissing': train_raw.nunique()})
display(schema)
print('Numeric features:', len(numeric_cols), '| Categorical features:', len(categorical_cols))
print('Duplicate complete rows:', train_raw.duplicated().sum())
print('Rows with any missing value:', train_raw.isna().any(axis=1).sum())
print('Missing feature cells:', train_raw.drop(columns=TARGET).isna().sum().sum())
display(train_raw[TARGET].value_counts(dropna=False).rename('count').to_frame())
display(train_raw[numeric_cols].describe().T.round(3))

q1, q3 = train_raw[numeric_cols].quantile(.25), train_raw[numeric_cols].quantile(.75)
iqr = q3 - q1
iqr_flags = ((train_raw[numeric_cols] < q1 - 1.5*iqr) |
             (train_raw[numeric_cols] > q3 + 1.5*iqr))
outlier_summary = pd.DataFrame({'IQR_flag_count': iqr_flags.sum(),
                                'IQR_flag_pct': iqr_flags.mean()*100})
display(outlier_summary.round(2))
schema.to_csv(OUT/'training_schema.csv')
outlier_summary.to_csv(OUT/'training_outlier_flags.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = train_raw[TARGET].value_counts().reindex(['no', 'yes'])
axes[0].bar(counts.index, counts.values, color=['#4C78A8', '#F58518'])
axes[0].set(title='Training class distribution', ylabel='Rows')
for i, count in enumerate(counts):
    axes[0].text(i, count, f'{count:,} ({count/counts.sum():.1%})', ha='center', va='bottom')
axes[0].set_ylim(0, counts.max()*1.16)
train_raw[numeric_cols].corr().pipe(lambda corr: axes[1].imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r'))
axes[1].set_xticks(range(len(numeric_cols)), numeric_cols, rotation=60, ha='right', fontsize=7)
axes[1].set_yticks(range(len(numeric_cols)), numeric_cols, fontsize=7)
axes[1].set_title('Training numeric Pearson correlations')
fig.colorbar(axes[1].images[0], ax=axes[1], fraction=.046, pad=.04)
fig.tight_layout()
fig.savefig(OUT/'training_overview.png')
plt.show()

## 3. Cleaning and feature decisions / 清洗与特征处理

- **Missing labels:** exclude from supervised fitting/scoring; never guess or impute the answer. Test rows with missing labels still receive predictions.
- **Numeric missing values:** median imputation inside each training fold. This is stable for skewed variables and avoids dropping otherwise useful rows.
- **Categorical missing values:** a dedicated `__MISSING__` category; existing `Unknown` and `Other` values remain valid observed categories because their semantics are undocumented.
- **Outliers:** retain them. IQR flags alone do not establish invalid observations. A tree model also provides a useful comparison with a linear model.
- **Features:** retain all 15 original predictors. Use one-hot encoding for nominal categories, avoiding artificial numeric order. Scale numerical features for Logistic Regression; trees do not require scaling. The correlated numeric features are retained with L2 regularization for the linear model; causal or feature-availability claims are not supported by a data dictionary.
- **No target-derived features, oversampling or test-driven selection.** Class weighting is evaluated by training-only cross-validation.

In [ ]:
train = train_raw.dropna(subset=[TARGET]).copy()
assert set(train[TARGET].unique()) == {'no', 'yes'}
assert train.duplicated().sum() == 0, 'Unexpected duplicate training rows; review before fitting.'
X_train = train.drop(columns=TARGET)
y_train = train[TARGET].map({'no': 0, 'yes': 1}).astype(int)
assert not np.isinf(X_train[numeric_cols].to_numpy()).any()
print('Labelled training rows:', len(train))
print('Excluded missing training labels:', train_raw[TARGET].isna().sum())
print(f'Positive prevalence: {y_train.mean():.4f}; majority baseline accuracy: {1-y_train.mean():.4f}')

def make_preprocessor(scale_numeric):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    categorical = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    return ColumnTransformer([
        ('numeric', Pipeline(numeric_steps), numeric_cols),
        ('categorical', categorical, categorical_cols)])

lr = Pipeline([('preprocess', make_preprocessor(True)),
               ('model', LogisticRegression(solver='lbfgs', max_iter=2000,
                                            tol=1e-4, random_state=SEED))])
rf = Pipeline([('preprocess', make_preprocessor(False)),
               ('model', RandomForestClassifier(n_estimators=200,
                    max_features='sqrt', random_state=SEED, n_jobs=1))])

## 4. Predeclared model comparison and tuning / 交叉验证调参
Logistic Regression is an interpretable linear decision model; Random Forest can capture nonlinearities and interactions. This is a comparison of model families, not an attempt to maximize leaderboard accuracy.

**Same 3 stratified folds**, shuffled with seed 42, for both models. Each fold fits its own imputer, encoder and (for LR) scaler. Select configurations by mean validation **average precision**, an area-style summary of the precision-recall curve (not trapezoidal PR-AUC). Class imbalance alone does not make PR-based evaluation universally preferable to ROC-based evaluation. Also report threshold-dependent F1/precision/recall and ROC-AUC.

- LR: `C ∈ {0.1, 1, 10}` × `class_weight ∈ {None, balanced}`: 6 configurations.
- RF: `max_depth ∈ {12, None}` × `min_samples_leaf ∈ {1, 5}` × `class_weight ∈ {None, balanced_subsample}`: 8 configurations.
- LR uses optimizer tolerance `1e-4` and a 2,000-iteration cap. The final LR iteration count is recorded. RF trains exactly 200 trees; nodes stop at depth/leaf/split/purity constraints. No validation early stopping is used.
- Both use probability threshold **0.5**, fixed before test evaluation. No threshold tuning.

42 CV fits plus two final refits. Runtime varies by CPU. Parallelism is limited to two jobs to keep Colab memory use modest. `n_jobs=1` inside RF avoids nested parallelism.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
grids = {
    'Logistic Regression': (lr, {'model__C': [.1, 1., 10.],
                               'model__class_weight': [None, 'balanced']}),
    'Random Forest': (rf, {'model__max_depth': [12, None],
                          'model__min_samples_leaf': [1, 5],
                          'model__class_weight': [None, 'balanced_subsample']})}
searches, cv_rows = {}, []
warnings.filterwarnings('error', category=ConvergenceWarning)
for name, (pipeline, grid) in grids.items():
    print('Tuning', name, flush=True)
    search = GridSearchCV(pipeline, grid, scoring={'ap': 'average_precision',
        'roc_auc': 'roc_auc', 'f1': 'f1'}, refit='ap', cv=cv,
        n_jobs=2, pre_dispatch=2, return_train_score=True, error_score='raise')
    start = time.perf_counter()
    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    searches[name] = search
    best = search.best_index_
    cv_rows.append({'model': name,
        'train_AP_mean': search.cv_results_['mean_train_ap'][best],
        'validation_AP_mean': search.best_score_,
        'validation_AP_sd': search.cv_results_['std_test_ap'][best],
        'validation_F1_mean': search.cv_results_['mean_test_f1'][best],
        'search_seconds': elapsed, 'best_parameters': search.best_params_})
    pd.DataFrame(search.cv_results_).to_csv(
        OUT/(name.lower().replace(' ', '_') + '_cv_results.csv'), index=False)
    print('Best parameters:', search.best_params_)
    print(f'Best CV AP: {search.best_score_:.4f}; time: {elapsed:.1f} seconds', flush=True)
cv_summary = pd.DataFrame(cv_rows)
display(cv_summary)
cv_summary.to_csv(OUT/'cv_summary.csv', index=False)
SELECTED_MODEL = max(searches, key=lambda name: searches[name].best_score_)
print('Model selected by training-only CV:', SELECTED_MODEL)
print('LR iterations:', searches['Logistic Regression'].best_estimator_.named_steps['model'].n_iter_)

## 5. Final held-out evaluation / 最终测试集评估
Both tuned models are refitted on all labelled training rows by `GridSearchCV`. The following cell opens the test labels for evaluation. It does not feed them back into model selection. The all-`no` majority rule is a diagnostic reference, not a third trained classifier.

In [ ]:
test_mask = test_raw[TARGET].notna()
test = test_raw.loc[test_mask].copy()
assert set(test[TARGET].unique()).issubset({'no', 'yes'})
X_test = test.drop(columns=TARGET)
y_test = test[TARGET].map({'no': 0, 'yes': 1}).astype(int)
assert not np.isinf(test_raw[numeric_cols].to_numpy()).any()
overlap = pd.util.hash_pandas_object(X_test, index=False).isin(
    pd.util.hash_pandas_object(X_train, index=False)).sum()
print('Labelled test rows:', len(test), '| Missing test labels:', (~test_mask).sum())
print('Test positive prevalence:', round(y_test.mean(), 4))
print('Exact feature-row train/test overlap:', overlap)
assert overlap == 0, 'Review overlap before interpreting test scores.'

def score_predictions(y, pred, proba):
    return {'accuracy': accuracy_score(y, pred),
            'balanced_accuracy': balanced_accuracy_score(y, pred),
            'precision_yes': precision_score(y, pred, zero_division=0),
            'recall_yes': recall_score(y, pred, zero_division=0),
            'f1_yes': f1_score(y, pred, zero_division=0),
            'roc_auc': roc_auc_score(y, proba),
            'average_precision': average_precision_score(y, proba)}

rows, probabilities, predictions, confusion_matrices = [], {}, {}, {}
baseline_proba = np.full(len(y_test), y_train.mean())
rows.append({'model': 'All-no reference', **score_predictions(
    y_test, np.zeros(len(y_test), dtype=int), baseline_proba)})
prediction_export = pd.DataFrame({'source_row_0based': test_raw.index,
                                   'true_label': test_raw[TARGET]})
for name, search in searches.items():
    fitted = search.best_estimator_
    positive_column = np.flatnonzero(fitted.classes_ == 1).item()
    all_proba = fitted.predict_proba(test_raw.drop(columns=TARGET))[:, positive_column]
    proba = all_proba[test_mask.to_numpy()]
    pred = (proba >= .5).astype(int)
    probabilities[name], predictions[name] = proba, pred
    rows.append({'model': name, **score_predictions(y_test, pred, proba)})
    confusion_matrices[name] = confusion_matrix(y_test, pred).tolist()
    print()
    print(name)
    print(classification_report(y_test, pred, target_names=['no', 'yes'], digits=4, zero_division=0))
    slug = name.lower().replace(' ', '_')
    prediction_export[slug+'_probability_yes'] = all_proba
    prediction_export[slug+'_label'] = np.where(all_proba >= .5, 'yes', 'no')

test_metrics = pd.DataFrame(rows).set_index('model')
display(test_metrics.round(4))
test_metrics.to_csv(OUT/'test_metrics.csv')
prediction_export.to_csv(OUT/'test_predictions.csv', index=False)
print('Training-CV choice remains:', SELECTED_MODEL)

## 6. Comparison plots / 图表
Confusion matrices expose false positives and false negatives. ROC-AUC measures ranking over false-positive/true-positive rates; the precision-recall curve highlights positive-class performance. Accuracy alone can obscure missed positives.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axes, searches):
    ConfusionMatrixDisplay.from_predictions(y_test, predictions[name],
        display_labels=['no', 'yes'], cmap='Blues', colorbar=False, ax=ax)
    ax.set_title(name + ' | threshold 0.5')
fig.tight_layout()
fig.savefig(OUT/'confusion_matrices.png')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, proba in probabilities.items():
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, proba, name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], '--', color='gray', label='Chance')
axes[1].axhline(y_test.mean(), linestyle='--', color='gray', label='Test prevalence')
axes[0].set_title('ROC curves')
axes[1].set_title('Precision-recall curves')
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT/'evaluation_curves.png')
plt.show()

## 7. Interpretation and reproducibility / 结果解释与导出
The selected configuration's CV score is optimistic as a performance estimate because it was used for selection; the untouched test split supplies the final estimate. Fold standard deviations describe variability, not a confidence interval or significance test.

Compare ranking (AP / ROC-AUC) separately from operating-point trade-offs (precision / recall / F1 at 0.5). Correlated predictors can make individual coefficient/importance interpretations unstable. There is no documented cost of false positives/negatives, group identifier, temporal order or feature collection time, so deployment utility, temporal generalization, fairness and causal conclusions are unverified. No feature-availability leakage was established or ruled out merely from column names.

This source copy has cleared outputs; running all cells regenerates the analysis. The aggregate results supplied alongside it come from the completed Google Colab run. Check the exported files against the report and add the actual repository URL before submission.

In [ ]:
summary = {
    'versions': VERSIONS, 'random_seed': SEED, 'source_sha256': SOURCE_HASHES,
    'train_rows_raw': len(train_raw), 'train_rows_labelled': len(train),
    'test_rows_raw': len(test_raw), 'test_rows_labelled': len(test),
    'training_class_counts': train[TARGET].value_counts().to_dict(),
    'test_class_counts': test[TARGET].value_counts().to_dict(),
    'numeric_features': numeric_cols, 'categorical_features': categorical_cols,
    'training_missing': train_raw.isna().sum().to_dict(),
    'training_IQR_flags': outlier_summary['IQR_flag_count'].to_dict(),
    'selected_by_training_cv': SELECTED_MODEL, 'threshold': .5,
    'cv_folds': 3, 'cv_summary': cv_rows,
    'test_metrics': test_metrics.reset_index().to_dict(orient='records'),
    'confusion_matrices': confusion_matrices,
    'encoded_feature_counts': {name: len(search.best_estimator_.named_steps[
        'preprocess'].get_feature_names_out()) for name, search in searches.items()}}
(OUT/'run_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
lr_row = test_metrics.loc['Logistic Regression']
rf_row = test_metrics.loc['Random Forest']
print(f'CV-selected model: {SELECTED_MODEL}')
print(f'Test AP: LR={lr_row.average_precision:.4f}, RF={rf_row.average_precision:.4f}')
print(f'Test F1: LR={lr_row.f1_yes:.4f}, RF={rf_row.f1_yes:.4f}')
print(f'Test recall: LR={lr_row.recall_yes:.4f}, RF={rf_row.recall_yes:.4f}')
print('Results saved to:', OUT.resolve())

## 8. Optional: download results / 可选下载
Set `DOWNLOAD_RESULTS = True` to download the results archive in Colab. The predictions remain in source-row order, including the unlabelled test row. Do not upload raw coursework data publicly without permission.

**GitHub:** upload this notebook, `README.md` and `requirements.txt` to your assignment repository; include its actual link in the PDF report. Run all cells to regenerate the outputs.

In [ ]:
DOWNLOAD_RESULTS = True
if DOWNLOAD_RESULTS:
    import shutil
    archive = shutil.make_archive('IN6227_results', 'zip', OUT)
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        print('Archive:', Path(archive).resolve())